In [ ]:
!pip install marker-pdf==0.3.10
!pip install texify==0.1.10

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 5.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of tabled-pdf to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 125.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.9/796.9 kB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.6/130.6 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 4

In [ ]:
!pip install "surya-ocr==0.6.13" "transformers==4.41.0" "tabled-pdf==0.1.4"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 65.5 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
texify 0.1.

In [ ]:
import os
import shutil
import json
from pathlib import Path
from google.colab import drive
from marker.convert import convert_single_pdf
from marker.models import load_all_models

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Paths - Adjusted for the "Books Preprocessing" Shortcut
# This assumes you added the shortcut directly to the root of your MyDrive
MASTER_FOLDER = Path("/content/drive/MyDrive/Books Preprocessing")
INPUT_DIR = MASTER_FOLDER / "books"
MARKDOWN_DIR = MASTER_FOLDER / "extracted_markdown"
DONE_DIR = MASTER_FOLDER / "books_done"

# Ensure output directories exist
for folder in [MARKDOWN_DIR, DONE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

def process_books():
    # 3. Load Models
    print("Loading AI Models...")
    models = load_all_models()
    print("Models loaded.")

    # 4. Get PDFs
    pdf_files = sorted([f for f in INPUT_DIR.iterdir() if f.suffix.lower() == ".pdf"])

    if not pdf_files:
        print(f"No PDFs found in {INPUT_DIR}. Check if the folder path is correct.")
        return

    print(f"Starting pipeline for {len(pdf_files)} books...")

    for i, pdf_path in enumerate(pdf_files, 1):
        try:
            # 5. Extract Book Name (e.g., "Build a Large Language Model")
            book_name = pdf_path.stem

            # Create subfolder for this specific book
            paper_folder = MARKDOWN_DIR / book_name
            paper_folder.mkdir(parents=True, exist_ok=True)

            print(f"\n[{i}/{len(pdf_files)}] Processing: {book_name}")

            # 6. Conversion Call
            full_text, images, metadata = convert_single_pdf(
                str(pdf_path),
                models,
                langs=["en"]
            )

            # 7. Save Logic
            # Save Markdown (.md)
            md_file_path = paper_folder / f"{book_name}.md"
            with open(md_file_path, "w", encoding="utf-8") as f:
                f.write(full_text)

            # Save Metadata (.json)
            meta_file_path = paper_folder / f"{book_name}_meta.json"
            with open(meta_file_path, "w", encoding="utf-8") as f:
                json.dump(metadata, f, indent=4)

            # Save Images
            if images:
                for img_name, img_obj in images.items():
                    img_obj.save(paper_folder / img_name)

            # 8. Move to 'books_done'
            shutil.move(str(pdf_path), str(DONE_DIR / pdf_path.name))
            print(f"Success. Moved {pdf_path.name} to books_done.")

        except Exception as e:
            print(f"Error on {pdf_path.name}: {e}")
            continue

if __name__ == "__main__":
    process_books()
    print("\nAll books processed successfully.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading AI Models...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loaded detection model vikp/surya_det3 on device cuda with dtype torch.float16
Loaded detection model vikp/surya_layout3 on device cuda with dtype torch.float16
Loaded reading order model vikp/surya_order on device cuda with dtype torch.float16
Loaded recognition model vikp/surya_rec2 on device cuda with dtype torch.float16
Loaded texify model to cuda with torch.float16 dtype
Loaded recognition model vikp/surya_tablerec on device cuda with dtype torch.float16
Models loaded.
Starting pipeline for 19 books...

[1/19] Processing: Building AI Agents with LLMs RAG and Knowledge Graphs


Detecting bboxes:  54%|█████▍    | 54/100 [02:27<01:59,  2.60s/it]

In [ ]:
# the coming cell is just for testing not need to run it

In [ ]:
import os
import shutil
from pathlib import Path
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Setup Paths
DRIVE_BASE = Path("/content/drive/MyDrive")
INPUT_DIR = DRIVE_BASE / "papers"
DONE_DIR = DRIVE_BASE / "papers_done"
MARKDOWN_DIR = DRIVE_BASE / "extracted_markdown"

In [ ]:
def count_items(folder_path, is_pdf=True):
    if not folder_path.exists():
        return 0
    if is_pdf:
        return len([f for f in folder_path.iterdir() if f.suffix.lower() == ".pdf"])
    else:
        # Count only directories in the extraction folder
        return len([f for f in folder_path.iterdir() if f.is_dir()])

print(f"📊 Full Pipeline Status:")
print(f"--------------------------")
print(f"📂 'papers' (To Process):         {count_items(INPUT_DIR, is_pdf=True)}")
print(f"✅ 'papers_done' (Moved):         {count_items(DONE_DIR, is_pdf=True)}")
print(f"📁 'extracted_markdown' (Folders): {count_items(MARKDOWN_DIR, is_pdf=False)}")

# Validation Logic
done_count = count_items(DONE_DIR, is_pdf=True)
folder_count = count_items(MARKDOWN_DIR, is_pdf=False)

if done_count == folder_count:
    print(f"\n✨ Sync Status: Perfect. Every moved PDF has a corresponding folder.")
else:
    print(f"\n⚠️ Sync Status: Mismatch. {abs(done_count - folder_count)} items are out of sync.")

In [ ]:
import os
import shutil
from pathlib import Path
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Setup Paths - Updated for the Shared Master Folder
MASTER_FOLDER = Path("/content/drive/MyDrive/Books Preprocessing")
INPUT_DIR = MASTER_FOLDER / "books"
DONE_DIR = MASTER_FOLDER / "books_done"
MARKDOWN_DIR = MASTER_FOLDER / "extracted_markdown"

In [ ]:
def count_items(folder_path, is_pdf=True):
    if not folder_path.exists():
        return 0
    if is_pdf:
        return len([f for f in folder_path.iterdir() if f.suffix.lower() == ".pdf"])
    else:
        # Count only directories in the extraction folder
        return len([f for f in folder_path.iterdir() if f.is_dir()])

In [ ]:
def print_status():
    print(f"\n📊 Books Pipeline Status:")
    print(f"--------------------------")
    print(f"📂 'books' (To Process):           {count_items(INPUT_DIR, is_pdf=True)}")
    print(f"✅ 'books_done' (Moved):           {count_items(DONE_DIR, is_pdf=True)}")
    print(f"📁 'extracted_markdown' (Folders): {count_items(MARKDOWN_DIR, is_pdf=False)}")

    # Validation Logic
    done_count = count_items(DONE_DIR, is_pdf=True)
    folder_count = count_items(MARKDOWN_DIR, is_pdf=False)

    if done_count == folder_count:
        print(f"✨ Sync Status: Perfect. Every moved Book has a corresponding folder.")
    else:
        print(f"⚠️ Sync Status: Mismatch. {abs(done_count - folder_count)} items are out of sync.")

In [ ]:
def reset_books():
    # Check if the folder exists
    if not DONE_DIR.exists():
        print(f"The folder {DONE_DIR} does not exist.")
        return

    # Ensure the target folder exists
    INPUT_DIR.mkdir(parents=True, exist_ok=True)

    # Gather all PDFs in books_done
    files_to_move = [f for f in DONE_DIR.iterdir() if f.suffix.lower() == ".pdf"]

    if not files_to_move:
        print("No PDF files found in 'books_done' to move back.")
        return

    print(f"Moving {len(files_to_move)} books back to 'books'...")

    for pdf_path in files_to_move:
        try:
            dest_path = INPUT_DIR / pdf_path.name
            shutil.move(str(pdf_path), str(dest_path))
            print(f"Moved: {pdf_path.name}")
        except Exception as e:
            print(f"Error moving {pdf_path.name}: {e}")

    print("\nReset complete. You can now run your updated extraction script.")

In [ ]:
# 1. Print current status
print_status()

In [ ]:
# 2. Run reset (moves PDFs back to 'books' folder)
# reset_books()

In [ ]:
# 3. Final status check after reset
# print_status()

In [ ]:
# --- DANGER ZONE ---
# WARNING: Only uncomment these if you want to completely WIPE the extraction progress
# shutil.rmtree(DONE_DIR, ignore_errors=True)
# shutil.rmtree(MARKDOWN_DIR, ignore_errors=True)